# Week 53, Day 3: One-dimensional DP

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w053d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
nums = [-2, 1, -3, 4, -1, 2, 1, -5, 4]
best_here, best = 0, nums[0]
history = []
for x in nums:
    best_here = max(x, best_here + x)
    best = max(best, best_here)
    history.append(best_here)
print(history)
print(best)

In [ ]:
def max_subarray(nums):
    """Kadane: the largest sum of a non-empty run, and where it is."""
    best_here, start = nums[0], 0
    best = (nums[0], 0, 0)
    for i in range(1, len(nums)):
        if best_here + nums[i] < nums[i]:
            best_here, start = nums[i], i          # start afresh here
        else:
            best_here += nums[i]                   # extend the run
        if best_here > best[0]:
            best = (best_here, start, i)
    return best

total, lo, hi = max_subarray([-2, 1, -3, 4, -1, 2, 1, -5, 4])
print(total, (lo, hi))

In [ ]:
def decodings(digits):
    """Ways to read a digit string as letters, 1 = A, ..., 26 = Z."""
    n = len(digits)
    ways = [1] + [0] * n                  # ways[i]: ways to read digits[:i]
    for i in range(1, n + 1):
        if digits[i - 1] != "0":
            ways[i] += ways[i - 1]          # the last piece is one digit
        if i >= 2 and 10 <= int(digits[i - 2:i]) <= 26:
            ways[i] += ways[i - 2]          # the last piece is two digits
    return ways[n]

print(decodings("12"), decodings("226"), decodings("06"), decodings("11106"))

In [ ]:
def max_profit(prices, fee=0):
    """Buy and sell any number of times, holding at most one share."""
    hold, free = float("-inf"), 0
    for p in prices:
        hold, free = max(hold, free - p), max(free, hold + p - fee)
    return free

print(max_profit([7, 1, 5, 3, 6, 4]), max_profit([1, 3, 2, 8, 4, 9], fee=2))

In [ ]:
def fewest_jumps(reach):
    """reach[i]: the longest jump from i. Fewest jumps from 0 to the end, or None."""
    n = len(reach)
    INF = float("inf")
    best = [INF] * n
    best[n - 1] = 0
    for i in range(n - 2, -1, -1):           # f(i) reads f(i + 1 ... i + reach[i])
        for j in range(i + 1, min(n, i + reach[i] + 1)):
            best[i] = min(best[i], 1 + best[j])
    return best[0] if best[0] < INF else None

print(fewest_jumps([2, 3, 1, 1, 4]), fewest_jumps([3, 2, 1, 0, 4]))

> **Key idea.** Four one-dimensional patterns cover most problems. Best ending here (Kadane) fixes the end of a contiguous run so f(i) depends only on f(i − 1); products need both the largest and smallest. Prefix splits let f(i) choose where the last piece starts, f(i) = best over j of f(j) plus the piece from j to i, in O(n²). Small state machines keep one value per state (holding, free, resting) and update them together. Reach problems take the best over the positions one move away. Match the wording of the question to the pattern, then write the state and recurrence.

### Exercise: Secret messages

A message of letters was encoded as numbers (A = 1, ..., Z = 26) and the numbers run together. Write `decodings(digits)`: how many letter strings it could be. Then `decode_all(digits)`: a sorted list of all of them (only for short inputs; build it with a prefix-split DP whose entries are lists of strings).

In [ ]:
def decodings(digits):
    ___

def decode_all(digits):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools\ndef brute(digits):\n    out = set()\n    def go(i, acc):\n        if i == len(digits):\n            out.add(acc)\n            return\n        for L in (1, 2):\n            piece = digits[i:i + L]\n            if len(piece) == L and piece[0] != \"0\" and 1 <= int(piece) <= 26:\n                go(i + L, acc + chr(64 + int(piece)))\n    go(0, \"\")\n    return sorted(out)\ndef test_decode():\n    \"decodings\"\n    assert decodings(\"12\") == 2 and decodings(\"226\") == 3 and decodings(\"06\") == 0 and decodings(\"\") == 1\n    assert decode_all(\"226\") == [\"BBF\", \"BZ\", \"VF\"] and decode_all(\"10\") == [\"J\"] and decode_all(\"30\") == []\n    for length in range(1, 6):\n        for digits in itertools.product(\"0126\", repeat=length):\n            d = \"\".join(digits)\n            assert decodings(d) == len(brute(d)) and decode_all(d) == brute(d), d\n    assert decodings(\"1\" * 80) == 37889062373143906")

### Exercise: The biggest product

Write `max_product(nums)`: the largest product of a non-empty contiguous run of whole numbers (which may be negative or zero), in one pass, keeping the largest **and** smallest product ending at each position.

In [ ]:
def max_product(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import math, random\ndef brute(nums):\n    return max(math.prod(nums[i:j]) for i in range(len(nums)) for j in range(i + 1, len(nums) + 1))\ndef test_product():\n    \"products\"\n    assert max_product([2, 3, -2, 4]) == 6 and max_product([-2, 0, -1]) == 0\n    assert max_product([-2, 3, -4]) == 24 and max_product([-3]) == -3\n    rng = random.Random(1)\n    for _ in range(300):\n        nums = [rng.randint(-4, 4) for _ in range(rng.randint(1, 9))]\n        assert max_product(nums) == brute(nums), nums")

### Exercise: Trading with rules

Write `max_profit(prices, fee=0, cooldown=False)`: the most profit from buying and selling one share at a time, any number of times, paying `fee` on each sale. With `cooldown=True`, you can't buy on the day after a sale. Use the states **hold**, **free** and **resting** (just sold), all updated together each day. Then `best_trades(prices, fee=0)`: the list of `(buy_day, sell_day)` pairs of one best plan without cooldown, by keeping, for each day and state, how you got there.

In [ ]:
def max_profit(prices, fee=0, cooldown=False):
    ___

def best_trades(prices, fee=0):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom functools import cache\ndef brute(prices, fee, cooldown):\n    n = len(prices)\n    @cache\n    def go(i, holding, can_buy):\n        if i == n:\n            return 0\n        best = go(i + 1, holding, True)\n        if holding:\n            best = max(best, prices[i] - fee + go(i + 1, False, not cooldown))\n        elif can_buy:\n            best = max(best, -prices[i] + go(i + 1, True, True))\n        return best\n    return go(0, False, True)\ndef test_stocks():\n    \"states\"\n    assert max_profit([7, 1, 5, 3, 6, 4]) == 7 and max_profit([1, 3, 2, 8, 4, 9], fee=2) == 8\n    assert max_profit([1, 2, 3, 0, 2], cooldown=True) == 3 and max_profit([]) == 0 and max_profit([5, 4, 3]) == 0\n    rng = random.Random(2)\n    for _ in range(200):\n        prices = [rng.randint(1, 15) for _ in range(rng.randint(0, 10))]\n        fee = rng.randint(0, 3)\n        for cd in (False, True):\n            assert max_profit(prices, fee, cd) == brute(prices, fee, cd), (prices, fee, cd)\n        trades = best_trades(prices, fee)\n        assert sum(prices[s] - prices[b] - fee for b, s in trades) == max_profit(prices, fee)\n        assert all(b < s for b, s in trades) and all(s1 < b2 for (_, s1), (b2, _) in zip(trades, trades[1:]))\n    assert best_trades([7, 1, 5, 3, 6, 4]) == [(1, 2), (3, 4)]")

### Exercise: Palindrome pieces

Write `min_cuts(s)`: the fewest cuts that split the string into pieces that are all palindromes (`"aab"` needs 1: `"aa" | "b"`). First fill a table `pal[i][j]` (is `s[i:j + 1]` a palindrome?) by growing outwards from each centre or by increasing length, so each check is O(1); then a prefix-split DP. Then `pieces(s)`: one split with the fewest pieces, as a list, found by walking back from the end and taking, each time, the **earliest** start j for the last piece that achieves the best count.

In [ ]:
def min_cuts(s):
    ___

def pieces(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(s):\n    n = len(s)\n    if n == 0:\n        return 0\n    best = n - 1\n    for cuts in range(n):\n        for where in itertools.combinations(range(1, n), cuts):\n            bounds = (0,) + where + (n,)\n            if all(s[a:b] == s[a:b][::-1] for a, b in zip(bounds, bounds[1:])):\n                return cuts\n    return best\ndef test_cuts():\n    \"palindromes\"\n    assert min_cuts(\"aab\") == 1 and min_cuts(\"a\") == 0 and min_cuts(\"\") == 0 and min_cuts(\"ab\") == 1\n    assert min_cuts(\"racecarxyz\") == 3 and pieces(\"aab\") == [\"aa\", \"b\"]\n    rng = random.Random(3)\n    for _ in range(150):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 10)))\n        assert min_cuts(s) == brute(s), s\n        p = pieces(s)\n        assert \"\".join(p) == s and all(x == x[::-1] for x in p) and len(p) == (min_cuts(s) + 1 if s else 0)\n    long = \"\".join(rng.choice(\"abc\") for _ in range(1500))\n    assert 0 < min_cuts(long) < 1500")

### Exercise: Stepping stones with costs

Stones stand in a row; landing on stone i costs `cost[i]` (the first stone is free; you start there). From any stone you may jump forward 1 to k stones. Write `cheapest_crossing(cost, k)`: the least total cost to land on the last stone, and `crossing_route(cost, k)`: the list of stones landed on, from 0 to the last (prefer the **nearest** previous stone on a tie).

In [ ]:
def cheapest_crossing(cost, k):
    ___

def crossing_route(cost, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(cost, k):\n    n = len(cost)\n    best = [float(\"inf\")]\n    def go(i, total):\n        if i == n - 1:\n            best[0] = min(best[0], total)\n            return\n        for j in range(i + 1, min(n, i + k + 1)):\n            go(j, total + cost[j])\n    go(0, 0)\n    return best[0]\ndef test_crossing():\n    \"reach with costs\"\n    assert cheapest_crossing([0, 5, 1, 4, 2], 2) == 3 and crossing_route([0, 5, 1, 4, 2], 2) == [0, 2, 4]\n    assert cheapest_crossing([0], 3) == 0 and crossing_route([0], 3) == [0]\n    rng = random.Random(4)\n    for _ in range(150):\n        n, k = rng.randint(1, 10), rng.randint(1, 4)\n        cost = [0] + [rng.randint(0, 9) for _ in range(n - 1)]\n        route = crossing_route(cost, k)\n        assert cheapest_crossing(cost, k) == brute(cost, k) == sum(cost[i] for i in route[1:])\n        assert route[0] == 0 and route[-1] == n - 1 and all(0 < b - a <= k for a, b in zip(route, route[1:]))")

### Bug hunt: A zero that reads as a letter

This decoder counts a lone `"0"` as a letter, so `"06"` gets one reading (and `"10"` two), and it prints `[1, 2]` instead of `[0, 1]`. A one-digit piece must not be `"0"`.

In [ ]:
def decodings(digits):
    n = len(digits)
    ways = [1] + [0] * n
    for i in range(1, n + 1):
        ways[i] += ways[i - 1]
        if i >= 2 and 10 <= int(digits[i - 2:i]) <= 26:
            ways[i] += ways[i - 2]
    return ways[n]

print([decodings("06"), decodings("10")])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no letter zero\"\n    expect_output(\"[0, 1]\")")

### Bug hunt: Forgetting the smallest

This keeps only the largest product ending here, so when a second negative number arrives it has nothing to multiply, and it prints `3` for `[-2, 3, -4]` instead of `24`. Keep the smallest product too, and update both together.

In [ ]:
def max_product(nums):
    high = best = nums[0]
    for x in nums[1:]:
        high = max(x, high * x)
        best = max(best, high)
    return best

print(max_product([-2, 3, -4]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"negatives flip\"\n    expect_output(\"24\")")

### Bug hunt: The empty prefix

This word-break table starts with `ok[0] = False`, but the empty prefix is always fine (nothing to split), so no word can ever start the text and it prints `False` for `"pineapple"`. Set the base case so it prints `True`.

In [ ]:
words = {"pine", "apple", "pineapple"}
text = "pineapple"
ok = [False] * (len(text) + 1)
for i in range(1, len(text) + 1):
    ok[i] = any(ok[j] and text[j:i] in words for j in range(i))
print(ok[len(text)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the base case\"\n    expect_output(\"True\")")

### Puzzle: Kadane's algorithm

Arrange the lines into Kadane's maximum subarray sum. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # Kadane's answer for [-1, -2, -3]
q = ___   # decodings("111")
r = ___   # the pattern for "fewest cuts into palindromes": "ending here", "last piece", "state machine" or "reach"
s = ___   # max_profit([1, 5, 2, 6]) with no fee and no cooldown
t = ___   # the running time of a prefix-split DP with O(1) piece checks, as "O(n)", "O(n log n)" or "O(n^2)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == -1 and q == 3 and r == \"last piece\", \"p, q and r\"\n    assert s == 8 and t == \"O(n^2)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: beautiful paragraphs

Week 35's text formatter filled each line greedily with as many words as fit. That can leave one line nearly empty after a tight one. A better rule, from Donald Knuth and Michael Plass's line-breaking work for TeX (1981), minimises the **total badness**: for each line except the last, the badness is the square of the spaces left over; the last line costs nothing. Words are separated by single spaces; no word is longer than the width.

- `greedy_lines(words, width)`: Week 35's greedy filling, as a list of lines (strings).
- `badness(lines, width)`: the total badness of a list of lines.
- `best_lines(words, width)`: lines with the least total badness, by a prefix-split DP over where the last line starts (f(i) = the least badness for the first i words; the piece from word j to word i − 1 costs nothing only when it is the paragraph's final line, i equal to the number of words). On a tie, prefer the **earliest** possible start for the last line.

Raise `ValueError` if a word is longer than the width.

In [ ]:
def greedy_lines(words, width):
    ___

def badness(lines, width):
    ___

def best_lines(words, width):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(words, width):\n    n = len(words)\n    best = None\n    for k in range(n):\n        for cuts in itertools.combinations(range(1, n), k):\n            b = (0,) + cuts + (n,)\n            lines = [\" \".join(words[x:y]) for x, y in zip(b, b[1:])]\n            if all(len(l) <= width for l in lines):\n                cost = sum((width - len(l)) ** 2 for l in lines[:-1])\n                best = cost if best is None else min(best, cost)\n    return best if n else 0\ndef test_lines():\n    \"paragraphs\"\n    words = \"aaa bb cc ddddd\".split()\n    assert greedy_lines(words, 6) == [\"aaa bb\", \"cc\", \"ddddd\"]\n    assert badness(greedy_lines(words, 6), 6) == 16\n    assert best_lines(words, 6) == [\"aaa\", \"bb cc\", \"ddddd\"] and badness(best_lines(words, 6), 6) == 10\n    assert best_lines([], 5) == [] and greedy_lines([], 5) == [] and badness([], 5) == 0\n    try:\n        best_lines([\"toolong\"], 5)\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(5)\n    for _ in range(80):\n        words = [\"x\" * rng.randint(1, 6) for _ in range(rng.randint(1, 9))]\n        width = rng.randint(6, 14)\n        lines = best_lines(words, width)\n        assert \" \".join(lines).split() == words and all(len(l) <= width for l in lines)\n        assert badness(lines, width) == brute(words, width) <= badness(greedy_lines(words, width), width)\n    text = (\"the quick brown fox jumps over the lazy dog \" * 60).split()\n    assert badness(best_lines(text, 30), 30) <= badness(greedy_lines(text, 30), 30)")

## Warm-up

`best_here` is the largest sum of a run of numbers **ending at the current position**. At each step the run either extends (add x) or starts afresh at x, whichever is bigger. The overall answer is the best of these.

## Best ending here

Many problems about **contiguous** runs (subarrays, substrings) define

**f(i) = the best answer for a run that ends exactly at position i**.

Then f(i) depends only on f(i − 1) (extend the run, or start a new one), and the answer is the maximum over all i. Fixing the end point is what makes it tractable: "the best run anywhere" has no simple recurrence, but "the best run ending here" does.

The maximum subarray version is **Kadane's algorithm** (Jay Kadane, 1984): O(n) time and O(1) memory, compared with Week 16's O(n log n) divide and conquer. It's worth seeing the pattern bend:

- **Maximum product**: a negative number turns the smallest product into the largest, so keep **both** the largest and smallest product ending here.
- **Longest run with a property**: f(i) = f(i − 1) + 1 if the property continues, else 1 (or 0).
- **With a cost to restart** or a limit on length, add it to the state.

## Where does the last piece start?

When a sequence is **cut into pieces**, define

**f(i) = the best way to handle the first i items** (the prefix of length i),

and decide where the **last piece** begins: f(i) = best over j < i of f(j) combined with the piece from j to i. f(0) is the empty prefix (cost 0, or 1 way). That's O(n) choices for each of n states, so O(n²) overall, times the cost of judging a piece.

Word break (Week 16's bonus, Week 52's remembered failures) is exactly this: f(i) is True if some j has f(j) True and `text[j:i]` a word. Others:

- **Decoding**: digits to letters (1 = A, ..., 26 = Z): the last piece is one digit or two.
- **Palindrome cuts**: the fewest cuts so every piece is a palindrome.
- **Line breaking**: split words into lines to minimise ugliness (today's challenge).

## Small state machines

Sometimes each position needs a little extra state: whether you're holding a share, whether you rested yesterday, which of three colours the last fence post was. Draw the states as a small diagram with the moves between them, and keep **one DP value per state**, updated together at each step.

Trading shares: on each day you either **hold** a share or are **free**. From free you may buy (pay the price) or wait; from hold you may sell (gain the price, minus a fee) or wait. With a **cooldown** after selling, add a third state, **resting**.

The tuple assignment matters: both new values must be computed from **yesterday's** values (Wednesday's bug hunt from yesterday).

## Reach

**Reach** problems ask how few steps get from the start to the end, where position i offers some moves: f(i) = 1 + the minimum of f over positions that can reach i (or that i can reach, filling backwards). With up to k moves from each position, that's O(n × k).

When every move costs the same, it's really BFS (Week 50) on a line, and a greedy "furthest reach" scan does it in O(n). DP earns its place when moves cost **different** amounts, or the cost depends on where you land.

## Which pattern?

| The question mentions | Try |
|---|---|
| a contiguous run (subarray, substring) | best ending here |
| cutting into pieces, segments, lines, words | where does the last piece start? |
| a condition that depends on the last step or two (holding, resting, colour) | a small state machine |
| fewest moves, steps or jumps along a line | reach |
| a subset of items with a budget | knapsack (Friday) |

When none fits, go back to yesterday's question: if you stopped halfway, what would you need to know?

## Practice

## Quiz

## Daily challenge

## Before you go

Find a problem you've solved before this week (in any week) and say which of today's four patterns it fits. Tomorrow: two dimensions, with grids.